In [45]:
# Imports
import pandas as pd 
import numpy as np
import os
import networkx

In [15]:
os.getcwd()

'/home/user/risk-propagation-elliptic/notebooks'

In [16]:
# PATHS TO DATA
BASE_PATH = "../data/elliptic_bitcoin_dataset/"
FEATS_PATH = f"{BASE_PATH}elliptic_txs_features.csv"
LABELS_PATH = f"{BASE_PATH}elliptic_txs_classes.csv"
EDGES_PATH = f"{BASE_PATH}elliptic_txs_edgelist.csv"

In [17]:
#Load the data

df_feats = pd.read_csv(FEATS_PATH, header=None)
df_labels = pd.read_csv(LABELS_PATH)
# These are the nodes of our graph.
# Class 1 is illicit, ckass 2 are licit, other ones are unknown
df_edges = pd.read_csv(EDGES_PATH)

In [18]:
df_feats.shape, df_labels.shape, df_edges.shape

((203769, 167), (203769, 2), (234355, 2))

In [19]:
#  column 0 is the id 
# column 1 is the timestamp one
df_feats.head()

,0,1,2,3,4,5,6,7,8,9,...,157,158,159,160,161,162,163,164,165,166
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117


In [21]:
df_labels.head()

,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown


In [22]:
df_edges.head()

,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206


In [ ]:
# Add the target to the features
df_feats_target = df_feats.merge(df_labels, right_on='txId', left_on=0)
df_feats_target["target"] = df_feats_target["class"].map({"2":1, "1":0})
df_feats_target.drop([0, "class"], axis=1, inplace=True)

/tmp/ipykernel_9811/1849664402.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_feats_target["target"] = df_feats_target["class"].map({"2":1, "1":0})


In [44]:
# Ignore ID label
df_describe = df_feats_target.iloc[:, :166].describe()
df_describe

,1,2,3,4,5,6,7,8,9,10,...,157,158,159,160,161,162,163,164,165,166
count,203769.000000,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05,...,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05,2.037690e+05
mean,23.843961,2.008513e-17,1.785345e-17,3.570689e-17,6.862418e-17,7.810883e-17,5.802370e-17,2.566433e-17,3.124353e-17,4.463361e-17,...,-2.789601e-17,1.612389e-16,-8.480387e-17,6.695042e-17,-4.686530e-17,-2.231681e-18,5.356034e-17,1.785345e-17,3.570689e-17,6.025538e-17
std,15.172170,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,...,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00
min,1.000000,-1.729826e-01,-2.105526e-01,-1.756361e+00,-1.219696e-01,-6.372457e-02,-1.130020e-01,-6.158379e-02,-1.636459e-01,-1.694603e-01,...,-5.770994e-01,-6.262286e-01,-9.790738e-01,-9.785560e-01,-2.160569e-01,-1.259391e-01,-1.311553e-01,-2.698175e-01,-1.760926e+00,-1.760984e+00
25%,9.000000,-1.725317e-01,-1.803266e-01,-1.201369e+00,-1.219696e-01,-4.387455e-02,-1.130020e-01,-6.158379e-02,-1.635168e-01,-1.690701e-01,...,-5.696264e-01,-5.946915e-01,-9.790738e-01,-9.785560e-01,-9.888874e-02,-8.749016e-02,-1.311553e-01,-1.405971e-01,-1.206134e-01,-1.197925e-01
50%,23.000000,-1.692045e-01,-1.328975e-01,4.636092e-01,-1.219696e-01,-4.387455e-02,-1.130020e-01,-6.158379e-02,-1.620440e-01,-1.662255e-01,...,-4.799511e-01,-4.559278e-01,2.411283e-01,2.414064e-01,1.827940e-02,-8.749016e-02,-1.311553e-01,-9.752359e-02,-1.206134e-01,-1.197925e-01
75%,38.000000,-1.318553e-01,-5.524241e-02,1.018602e+00,-1.219696e-01,-4.387455e-02,-1.130020e-01,-6.158379e-02,-1.355932e-01,-1.323665e-01,...,1.552495e-01,1.212026e-01,1.305594e+00,1.398764e+00,1.827940e-02,-8.749016e-02,-8.467423e-02,-9.752359e-02,1.520067e-01,1.199710e-01
max,49.000000,7.168197e+01,7.359505e+01,2.683580e+00,4.902760e+01,2.600907e+02,5.456518e+01,1.134409e+02,7.335457e+01,7.231840e+01,...,7.862953e+00,7.914041e+00,1.461330e+00,1.461369e+00,1.170692e+02,2.518490e+02,2.387835e+02,1.057340e+02,1.519700e+00,1.521399e+00


In [ ]:
# No na
df_feats_target.isna().sum().sort_values()

1              0
2              0
3              0
4              0
5              0
           ...  
164            0
165            0
166            0
txId           0
target    157205
Length: 168, dtype: int64

In [ ]:
# Some timestamps are riskier
df_feats_target.groupby(1)["target"].value_counts().sort_values()

1   target
46  0.0          2
6   0.0          5
45  0.0          5
5   0.0          8
3   0.0         11
              ... 
22  1.0       1605
36  1.0       1675
5   1.0       1874
42  1.0       1915
1   1.0       2130
Name: count, Length: 98, dtype: int64

In [64]:
# No constant columns
df_describe.loc[:, df_describe.loc["std"] < 0.1]

""
count
mean
std
min
25%
50%
75%
max


In [ ]:
IQR = (df_describe.loc["75%", :] - df_describe.loc["25%", :])
outliers = (
    (df_feats_target.iloc[:, :166] < (df_describe.loc["25%", :] - 2*IQR))
    | (df_feats_target.iloc[:, :166] > (df_describe.loc["75%", :] + 2*IQR))
) 

In [91]:
# Too many outliers, need a different approach
df_feats_target.iloc[:, :166][outliers.any(axis=1)]

,1,2,3,4,5,6,7,8,9,10,...,157,158,159,160,161,162,163,164,165,166
2,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,-0.168576,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,-0.115831,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,0.041399,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117
5,1,0.961040,-0.081127,-1.201369,1.303743,0.333276,1.480381,-0.061584,-0.163577,0.038305,...,-0.504702,-0.422589,-0.226790,-0.117629,0.018279,0.277775,0.413931,1.149556,-0.696053,-0.695540
6,1,-0.171264,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.161887,-0.167726,...,-0.569626,-0.607306,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203764,49,-0.145771,-0.163752,0.463609,-0.121970,-0.043875,-0.113002,-0.061584,-0.135803,-0.142008,...,-0.577099,-0.613614,0.241128,0.241406,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
203765,49,-0.165920,-0.123607,1.018602,-0.121970,-0.043875,-0.113002,-0.061584,-0.156418,-0.162334,...,0.162722,0.010822,1.461330,1.461369,-0.098889,-0.087490,-0.084674,-0.140597,-1.760926,-1.760984
203766,49,-0.172014,-0.078182,1.018602,0.028105,-0.043875,0.054722,-0.061584,-0.163626,-0.168778,...,1.261246,1.985050,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
203767,49,-0.172842,-0.176622,1.018602,-0.121970,-0.043875,-0.113002,-0.061584,-0.163501,-0.169317,...,-0.397749,-0.411776,1.461330,1.461369,-0.098889,-0.087490,-0.084674,-0.140597,1.519700,1.521399


In [95]:
upper = df_feats_target.iloc[:, :166].quantile(0.999)
lower = df_feats_target.iloc[:, :166].quantile(0.001)
outliers_new = (df_feats_target.iloc[:, :166] > upper) | (df_feats_target.iloc[:, :166] < lower) 

In [ ]:
# Still, too many outliers, feels like this is the norm in this fraud context.
df_feats_target.iloc[:, :166][outliers_new.any(axis=1)]


,1,2,3,4,5,6,7,8,9,10,...,157,158,159,160,161,162,163,164,165,166
3,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,-0.115831,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
16,1,-0.172306,-0.184668,-1.201369,0.028105,-0.043875,-0.029140,0.242712,-0.163640,-0.169115,...,-0.577099,-0.600999,0.241128,0.241406,0.018279,-0.068266,-0.084674,-0.054450,-1.760926,-1.760984
27,1,-0.172350,-0.155133,0.463609,0.328255,-0.043875,0.390171,-0.061584,-0.163645,-0.169226,...,-0.569626,-0.582077,1.461330,1.461369,0.252616,0.066306,0.147731,0.333211,-1.760926,-1.760984
47,1,-0.134443,0.527313,1.573595,28.317246,-0.043875,31.670786,-0.061584,-0.163634,-0.164965,...,2.165473,1.890439,1.461330,1.461369,-0.098889,-0.087490,-0.084674,-0.140597,-1.760926,-1.760984
52,1,-0.172290,-0.184668,-1.201369,-0.046932,-0.043875,-0.029140,-0.061584,-0.163582,-0.168824,...,-0.539735,-0.569462,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203733,49,-0.172954,-0.149127,0.463609,-0.121970,-0.063725,-0.113002,-0.061584,-0.163615,-0.169430,...,-0.577099,0.647874,0.241128,0.241406,7.165536,1.085202,-0.131155,5.157442,-0.120613,-0.119792
203735,49,-0.172975,-0.081127,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.163635,-0.169449,...,2.860457,2.678868,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
203754,49,-0.172962,-0.126566,1.018602,-0.121970,-0.063725,-0.113002,-0.061584,-0.163622,-0.169437,...,-0.577099,0.647874,0.241128,0.241406,10.914916,1.700384,-0.131155,7.914145,-0.120613,-0.119792
203762,49,-0.172974,-0.156732,1.018602,-0.121970,-0.043875,-0.113002,-0.061584,-0.163636,-0.169451,...,-0.577099,0.647874,0.241128,0.241406,7.165536,1.085202,-0.131155,5.157442,-0.120613,-0.119792
